In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
======================================================================
ANÁLISIS DE IRRADIANCIA - SISTEMA AUTÓNOMO FOTOVOLTAICO
======================================================================

Columnas esperadas en los archivos .dat:

1. Tiempo
2. Irradiancia
3. Temperatura ambiente
4. Temperatura de sistemas de operación
5. Voltaje módulo PV
6. Voltaje batería

El programa analiza:

- Irradiancia máxima y mínima
- Tiempo del máximo
- Intervalo de irradiancia significativa
- Duración del intervalo significativo
- Irradiancia media
- Desviación estándar
- Coeficiente de variación
- Pendiente antes y después del máximo
- Variabilidad alrededor del máximo
- Número de máximos locales relevantes
- Forma aproximada de la curva
- Generación de gráfica de irradiancia vs. tiempo
- Generación de informe .txt
- Generación de resultados .csv

======================================================================
"""

import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# =====================================================================
# CONFIGURACIÓN
# =====================================================================

# Carpeta donde se encuentran los archivos .dat
CARPETA_DATOS = "."

# Carpeta donde se guardarán los resultados
CARPETA_RESULTADOS = "resultados_irradiancia"

# Umbral para definir irradiancia significativa
# 0.10 = 10 % del máximo diario
UMBRAL_SIGNIFICATIVO = 0.10

# Número mínimo de puntos consecutivos para considerar
# que existe irradiancia significativa.
MIN_PUNTOS_SIGNIFICATIVOS = 3

# Ventana temporal alrededor del máximo para estudiar
# su comportamiento.
# Se expresa en minutos.
VENTANA_MAXIMO_MIN = 30


# =====================================================================
# NOMBRES DE LAS COLUMNAS
# =====================================================================

COLUMNAS = [
    "tiempo",
    "irradiancia",
    "temperatura_ambiente",
    "temperatura_sistemas",
    "voltaje_pv",
    "voltaje_bateria"
]


# =====================================================================
# CREAR CARPETA DE RESULTADOS
# =====================================================================

os.makedirs(CARPETA_RESULTADOS, exist_ok=True)


# =====================================================================
# FUNCIÓN PARA CONVERTIR EL TIEMPO
# =====================================================================

def convertir_tiempo(serie):
    """
    Intenta interpretar automáticamente el tiempo.

    Se admiten formatos como:

    08:30:00
    08:30
    2026-08-20 08:30:00

    Si no puede convertirse, se devuelve la serie original.
    """

    tiempo = pd.to_datetime(
        serie,
        errors="coerce",
        dayfirst=True
    )

    return tiempo


# =====================================================================
# CARGAR ARCHIVO
# =====================================================================

def cargar_datos(archivo):

    print(f"\nLeyendo: {archivo}")

    # --------------------------------------------------------------
    # Primero intentamos detectar automáticamente el separador.
    # --------------------------------------------------------------

    try:

        datos = pd.read_csv(
            archivo,
            sep=None,
            engine="python",
            comment="#"
        )

    except Exception:

        # Si falla, asumimos espacios o tabulaciones
        datos = pd.read_csv(
            archivo,
            sep=r"\s+",
            engine="python",
            comment="#",
            header=None
        )

    # --------------------------------------------------------------
    # Si solamente tenemos 6 columnas, asignamos los nombres.
    # --------------------------------------------------------------

    if len(datos.columns) == 6:

        datos.columns = COLUMNAS

    else:

        # Intentamos leer directamente sin encabezado
        datos = pd.read_csv(
            archivo,
            sep=r"\s+|,|;",
            engine="python",
            comment="#",
            header=None
        )

        if len(datos.columns) != 6:

            raise ValueError(
                f"El archivo {archivo} no contiene exactamente "
                f"6 columnas."
            )

        datos.columns = COLUMNAS

    # --------------------------------------------------------------
    # Convertir tiempo
    # --------------------------------------------------------------

    datos["tiempo"] = convertir_tiempo(datos["tiempo"])

    # --------------------------------------------------------------
    # Convertir variables numéricas
    # --------------------------------------------------------------

    for columna in COLUMNAS[1:]:

        datos[columna] = pd.to_numeric(
            datos[columna],
            errors="coerce"
        )

    # --------------------------------------------------------------
    # Eliminar filas inválidas
    # --------------------------------------------------------------

    datos = datos.dropna(
        subset=["tiempo", "irradiancia"]
    ).copy()

    # Ordenar temporalmente

    datos = datos.sort_values("tiempo")

    datos = datos.reset_index(drop=True)

    return datos


# =====================================================================
# DETECTAR INTERVALO DE IRRADIANCIA SIGNIFICATIVA
# =====================================================================

def intervalo_significativo(datos):

    G = datos["irradiancia"].values

    Gmax = np.max(G)

    umbral = UMBRAL_SIGNIFICATIVO * Gmax

    mascara = G >= umbral

    # --------------------------------------------------------------
    # Buscar grupos consecutivos de valores verdaderos.
    # --------------------------------------------------------------

    grupos = []

    inicio = None

    for i, valor in enumerate(mascara):

        if valor and inicio is None:

            inicio = i

        elif not valor and inicio is not None:

            fin = i - 1

            if fin - inicio + 1 >= MIN_PUNTOS_SIGNIFICATIVOS:

                grupos.append((inicio, fin))

            inicio = None

    # Si termina siendo significativo
    if inicio is not None:

        fin = len(mascara) - 1

        if fin - inicio + 1 >= MIN_PUNTOS_SIGNIFICATIVOS:

            grupos.append((inicio, fin))

    if not grupos:

        return None, umbral

    # --------------------------------------------------------------
    # Seleccionar el intervalo significativo que contiene
    # el máximo global.
    # --------------------------------------------------------------

    indice_max = np.argmax(G)

    for inicio, fin in grupos:

        if inicio <= indice_max <= fin:

            return (inicio, fin), umbral

    # Si el máximo no cae dentro de ninguno, usar el mayor grupo

    grupo = max(
        grupos,
        key=lambda x: x[1] - x[0]
    )

    return grupo, umbral


# =====================================================================
# ANÁLISIS ALREDEDOR DEL MÁXIMO
# =====================================================================

def analizar_maximo(datos):

    G = datos["irradiancia"].values
    t = datos["tiempo"]

    indice_max = np.argmax(G)

    Gmax = G[indice_max]
    tiempo_max = t.iloc[indice_max]

    # --------------------------------------------------------------
    # Ventana temporal
    # --------------------------------------------------------------

    tiempo_inicio = tiempo_max - pd.Timedelta(
        minutes=VENTANA_MAXIMO_MIN
    )

    tiempo_fin = tiempo_max + pd.Timedelta(
        minutes=VENTANA_MAXIMO_MIN
    )

    ventana = datos[
        (datos["tiempo"] >= tiempo_inicio) &
        (datos["tiempo"] <= tiempo_fin)
    ].copy()

    if len(ventana) < 3:

        return {
            "pendiente_antes": np.nan,
            "pendiente_despues": np.nan,
            "variabilidad_maximo": np.nan,
            "num_fluctuaciones": np.nan
        }

    # --------------------------------------------------------------
    # Separar antes y después del máximo
    # --------------------------------------------------------------

    antes = ventana[
        ventana["tiempo"] <= tiempo_max
    ]

    despues = ventana[
        ventana["tiempo"] >= tiempo_max
    ]

    # --------------------------------------------------------------
    # Convertir tiempo a segundos
    # --------------------------------------------------------------

    def tiempo_segundos(serie):

        return (
            serie - tiempo_max
        ).dt.total_seconds().values

    pendiente_antes = np.nan
    pendiente_despues = np.nan

    if len(antes) >= 2:

        x = tiempo_segundos(antes["tiempo"])
        y = antes["irradiancia"].values

        pendiente_antes = np.polyfit(
            x,
            y,
            1
        )[0]

    if len(despues) >= 2:

        x = tiempo_segundos(despues["tiempo"])
        y = despues["irradiancia"].values

        pendiente_despues = np.polyfit(
            x,
            y,
            1
        )[0]

    # --------------------------------------------------------------
    # Variabilidad alrededor del máximo
    # --------------------------------------------------------------

    desviacion = ventana["irradiancia"].std()

    # --------------------------------------------------------------
    # Fluctuaciones alrededor del máximo
    # --------------------------------------------------------------

    diferencias = np.diff(
        ventana["irradiancia"].values
    )

    cambios_signo = np.diff(
        np.sign(diferencias)
    )

    num_fluctuaciones = np.sum(
        cambios_signo != 0
    )

    return {
        "pendiente_antes": pendiente_antes,
        "pendiente_despues": pendiente_despues,
        "variabilidad_maximo": desviacion,
        "num_fluctuaciones": int(num_fluctuaciones)
    }


# =====================================================================
# CLASIFICACIÓN DESCRIPTIVA DE LA FORMA
# =====================================================================

def clasificar_curva(datos):

    G = datos["irradiancia"].values

    Gmax = np.max(G)
    Gmean = np.mean(G)
    Gstd = np.std(G)

    if Gmean == 0:

        return "Sin irradiancia significativa"

    coef_variacion = Gstd / Gmean

    # --------------------------------------------------------------
    # Número de máximos locales
    # --------------------------------------------------------------

    derivada = np.diff(G)

    signos = np.sign(derivada)

    cambios = np.diff(signos)

    maximos_locales = np.sum(
        cambios < 0
    )

    # --------------------------------------------------------------
    # Clasificación aproximada
    # --------------------------------------------------------------

    if maximos_locales <= 2 and coef_variacion < 0.8:

        return "Perfil diario relativamente suave"

    elif maximos_locales > 2 and coef_variacion < 1.0:

        return "Perfil con fluctuaciones"

    elif coef_variacion >= 1.0:

        return "Perfil altamente variable"

    else:

        return "Perfil irregular"


# =====================================================================
# ANÁLISIS PRINCIPAL
# =====================================================================

def analizar_archivo(archivo):

    datos = cargar_datos(archivo)

    G = datos["irradiancia"]

    # --------------------------------------------------------------
    # Estadísticos principales
    # --------------------------------------------------------------

    Gmax = G.max()
    Gmin = G.min()
    Gmean = G.mean()
    Gstd = G.std()

    indice_max = G.idxmax()

    tiempo_max = datos.loc[
        indice_max,
        "tiempo"
    ]

    # --------------------------------------------------------------
    # Irradiancia significativa
    # --------------------------------------------------------------

    intervalo, umbral = intervalo_significativo(datos)

    if intervalo is not None:

        inicio, fin = intervalo

        tiempo_inicio = datos.loc[
            inicio,
            "tiempo"
        ]

        tiempo_fin = datos.loc[
            fin,
            "tiempo"
        ]

        duracion = (
            tiempo_fin - tiempo_inicio
        ).total_seconds() / 3600

    else:

        tiempo_inicio = None
        tiempo_fin = None
        duracion = np.nan

    # --------------------------------------------------------------
    # Análisis del máximo
    # --------------------------------------------------------------

    analisis_max = analizar_maximo(datos)

    # --------------------------------------------------------------
    # Forma de la curva
    # --------------------------------------------------------------

    forma = clasificar_curva(datos)

    # --------------------------------------------------------------
    # Coeficiente de variación
    # --------------------------------------------------------------

    if Gmean != 0:

        coef_variacion = Gstd / Gmean

    else:

        coef_variacion = np.nan

    # --------------------------------------------------------------
    # RESULTADOS
    # --------------------------------------------------------------

    resultados = {

        "archivo": os.path.basename(archivo),

        "G_max": Gmax,

        "G_min": Gmin,

        "G_promedio": Gmean,

        "G_desviacion_std": Gstd,

        "coeficiente_variacion": coef_variacion,

        "tiempo_G_max": tiempo_max,

        "umbral_significativo": umbral,

        "inicio_irradiancia_significativa":
            tiempo_inicio,

        "fin_irradiancia_significativa":
            tiempo_fin,

        "duracion_irradiancia_significativa_h":
            duracion,

        "pendiente_antes_maximo":
            analisis_max["pendiente_antes"],

        "pendiente_despues_maximo":
            analisis_max["pendiente_despues"],

        "desviacion_ventana_maximo":
            analisis_max["variabilidad_maximo"],

        "fluctuaciones_alrededor_maximo":
            analisis_max["num_fluctuaciones"],

        "forma_curva":
            forma
    }

    # --------------------------------------------------------------
    # GENERAR GRÁFICA
    # --------------------------------------------------------------

    nombre_base = os.path.splitext(
        os.path.basename(archivo)
    )[0]

    figura = plt.figure(
        figsize=(12, 7)
    )

    plt.plot(
        datos["tiempo"],
        datos["irradiancia"],
        linewidth=1.2,
        label="Irradiancia global"
    )

    # --------------------------------------------------------------
    # Máximo
    # --------------------------------------------------------------

    plt.scatter(
        [tiempo_max],
        [Gmax],
        s=60,
        zorder=5,
        label=f"Máximo = {Gmax:.2f}"
    )

    plt.annotate(
        f"Máximo\n{Gmax:.2f}",
        xy=(tiempo_max, Gmax),
        xytext=(10, 15),
        textcoords="offset points"
    )

    # --------------------------------------------------------------
    # Umbral significativo
    # --------------------------------------------------------------

    plt.axhline(
        umbral,
        linestyle="--",
        linewidth=1,
        label=f"Umbral = {umbral:.2f}"
    )

    # --------------------------------------------------------------
    # Intervalo significativo
    # --------------------------------------------------------------

    if intervalo is not None:

        plt.axvspan(
            tiempo_inicio,
            tiempo_fin,
            alpha=0.15,
            label="Intervalo significativo"
        )

    # --------------------------------------------------------------
    # Ventana alrededor del máximo
    # --------------------------------------------------------------

    plt.axvspan(
        tiempo_max - pd.Timedelta(
            minutes=VENTANA_MAXIMO_MIN
        ),
        tiempo_max + pd.Timedelta(
            minutes=VENTANA_MAXIMO_MIN
        ),
        alpha=0.08,
        label=f"±{VENTANA_MAXIMO_MIN} min alrededor del máximo"
    )

    plt.xlabel("Tiempo")
    plt.ylabel("Irradiancia global")

    plt.title(
        f"Irradiancia global - {nombre_base}"
    )

    plt.grid(
        True,
        alpha=0.3
    )

    plt.legend()

    plt.gcf().autofmt_xdate()

    plt.tight_layout()

    archivo_figura = os.path.join(
        CARPETA_RESULTADOS,
        f"{nombre_base}_irradiancia.png"
    )

    plt.savefig(
        archivo_figura,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()

    # --------------------------------------------------------------
    # GUARDAR INFORME TXT
    # --------------------------------------------------------------

    archivo_txt = os.path.join(
        CARPETA_RESULTADOS,
        f"{nombre_base}_analisis.txt"
    )

    with open(
        archivo_txt,
        "w",
        encoding="utf-8"
    ) as f:

        f.write(
            "====================================================\n"
        )

        f.write(
            "ANÁLISIS DE IRRADIANCIA\n"
        )

        f.write(
            "====================================================\n\n"
        )

        f.write(
            f"Archivo: {nombre_base}\n\n"
        )

        f.write(
            "ESTADÍSTICOS DE IRRADIANCIA\n"
        )

        f.write(
            "----------------------------------------------------\n"
        )

        f.write(
            f"Irradiancia máxima: "
            f"{Gmax:.4f}\n"
        )

        f.write(
            f"Irradiancia mínima: "
            f"{Gmin:.4f}\n"
        )

        f.write(
            f"Irradiancia promedio: "
            f"{Gmean:.4f}\n"
        )

        f.write(
            f"Desviación estándar: "
            f"{Gstd:.4f}\n"
        )

        f.write(
            f"Coeficiente de variación: "
            f"{coef_variacion:.4f}\n\n"
        )

        f.write(
            "MÁXIMO DE IRRADIANCIA\n"
        )

        f.write(
            "----------------------------------------------------\n"
        )

        f.write(
            f"Tiempo del máximo: "
            f"{tiempo_max}\n"
        )

        f.write(
            f"Valor máximo: "
            f"{Gmax:.4f}\n\n"
        )

        f.write(
            "IRRADIANCIA SIGNIFICATIVA\n"
        )

        f.write(
            "----------------------------------------------------\n"
        )

        f.write(
            f"Umbral utilizado: "
            f"{umbral:.4f}\n"
        )

        f.write(
            f"Porcentaje del máximo: "
            f"{UMBRAL_SIGNIFICATIVO * 100:.1f}%\n"
        )

        f.write(
            f"Inicio: "
            f"{tiempo_inicio}\n"
        )

        f.write(
            f"Fin: "
            f"{tiempo_fin}\n"
        )

        f.write(
            f"Duración: "
            f"{duracion:.3f} horas\n\n"
        )

        f.write(
            "COMPORTAMIENTO ALREDEDOR DEL MÁXIMO\n"
        )

        f.write(
            "----------------------------------------------------\n"
        )

        f.write(
            f"Ventana analizada: "
            f"±{VENTANA_MAXIMO_MIN} minutos\n"
        )

        f.write(
            f"Pendiente antes del máximo: "
            f"{analisis_max['pendiente_antes']:.6f}\n"
        )

        f.write(
            f"Pendiente después del máximo: "
            f"{analisis_max['pendiente_despues']:.6f}\n"
        )

        f.write(
            f"Desviación alrededor del máximo: "
            f"{analisis_max['variabilidad_maximo']:.4f}\n"
        )

        f.write(
            f"Fluctuaciones detectadas: "
            f"{analisis_max['num_fluctuaciones']}\n\n"
        )

        f.write(
            "FORMA DE LA CURVA\n"
        )

        f.write(
            "----------------------------------------------------\n"
        )

        f.write(
            f"{forma}\n"
        )

    return resultados


# =====================================================================
# PROCESAR TODOS LOS ARCHIVOS .DAT
# =====================================================================

def main():

    archivos = sorted(
        glob.glob(
            os.path.join(
                CARPETA_DATOS,
                "*.dat"
            )
        )
    )

    if not archivos:

        print(
            "\nNo se encontraron archivos .dat "
            "en la carpeta indicada."
        )

        return

    print(
        f"\nSe encontraron {len(archivos)} archivos .dat."
    )

    todos_resultados = []

    for archivo in archivos:

        try:

            resultado = analizar_archivo(
                archivo
            )

            todos_resultados.append(
                resultado
            )

            print(
                f"OK: {os.path.basename(archivo)}"
            )

        except Exception as error:

            print(
                f"ERROR en {archivo}: {error}"
            )

    # --------------------------------------------------------------
    # Guardar resumen general
    # --------------------------------------------------------------

    if todos_resultados:

        resumen = pd.DataFrame(
            todos_resultados
        )

        archivo_resumen = os.path.join(
            CARPETA_RESULTADOS,
            "resumen_irradiancia.csv"
        )

        resumen.to_csv(
            archivo_resumen,
            index=False,
            encoding="utf-8"
        )

        print(
            "\n===================================================="
        )

        print(
            "ANÁLISIS TERMINADO"
        )

        print(
            "===================================================="
        )

        print(
            f"Resultados guardados en:\n"
            f"{CARPETA_RESULTADOS}/"
        )

        print(
            f"\nResumen general:\n"
            f"{archivo_resumen}"
        )


# =====================================================================
# EJECUCIÓN
# =====================================================================

if __name__ == "__main__":

    main()


Se encontraron 21 archivos .dat.

Leyendo: ./2026-07-11.dat


/tmp/ipykernel_5252/125518520.py:107: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  tiempo = pd.to_datetime(


OK: 2026-07-11.dat

Leyendo: ./2026-07-15.dat


/tmp/ipykernel_5252/125518520.py:107: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  tiempo = pd.to_datetime(


OK: 2026-07-15.dat

Leyendo: ./2026-07-16.dat


...

OK: 2026-08-14.dat

Leyendo: ./2026-08-15.dat


/tmp/ipykernel_5252/125518520.py:107: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  tiempo = pd.to_datetime(


OK: 2026-08-15.dat

Leyendo: ./2026-08-16.dat


/tmp/ipykernel_5252/125518520.py:107: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  tiempo = pd.to_datetime(


OK: 2026-08-16.dat

ANÁLISIS TERMINADO
Resultados guardados en:
resultados_irradiancia/

Resumen general:
resultados_irradiancia/resumen_irradiancia.csv


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
======================================================================
ANÁLISIS DE IRRADIANCIA - SISTEMA AUTÓNOMO FOTOVOLTAICO
======================================================================

Lee archivos .dat con las siguientes columnas:

1. Tiempo
2. Irradiancia
3. Temperatura ambiente
4. Temperatura de sistemas de operación
5. Voltaje módulo PV
6. Voltaje batería

Para cada archivo calcula:

- Irradiancia promedio de todo el registro
- Irradiancia promedio durante el periodo significativo
- Irradiancia máxima
- Irradiancia mínima
- Hora del máximo
- Umbral de irradiancia significativa
- Inicio del periodo significativo
- Fin del periodo significativo
- Duración del periodo significativo
- Desviación estándar
- Coeficiente de variación
- Clasificación del día

Clasificación indicada por el director:

0 <= irradiancia promedio < 200 W/m²  -> Nublado
200 <= irradiancia promedio <= 400 W/m² -> Seminublado
irradiancia promedio > 400 W/m² -> Claro

======================================================================
"""

import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# =====================================================================
# CONFIGURACIÓN
# =====================================================================

# Carpeta donde están los .dat
CARPETA_DATOS = "."

# Carpeta donde se guardarán los resultados
CARPETA_RESULTADOS = "analisis_irradiancia"

# Porcentaje utilizado para determinar irradiancia significativa
# 0.10 = 10 % del máximo
UMBRAL_SIGNIFICATIVO = 0.10

# Número mínimo de puntos consecutivos por encima del umbral
MIN_PUNTOS_SIGNIFICATIVOS = 3


# =====================================================================
# COLUMNAS
# =====================================================================

COLUMNAS = [
    "tiempo",
    "irradiancia",
    "temperatura_ambiente",
    "temperatura_sistemas",
    "voltaje_pv",
    "voltaje_bateria"
]


# =====================================================================
# CREAR CARPETA DE RESULTADOS
# =====================================================================

os.makedirs(CARPETA_RESULTADOS, exist_ok=True)


# =====================================================================
# FUNCIÓN PARA LEER EL TIEMPO
# =====================================================================

def convertir_tiempo(serie):

    """
    Convierte la columna de tiempo a formato datetime.
    """

    tiempo = pd.to_datetime(
        serie,
        errors="coerce",
        dayfirst=True
    )

    return tiempo


# =====================================================================
# LEER ARCHIVO .DAT
# =====================================================================

def leer_archivo(archivo):

    """
    Lee un archivo .dat.

    Se intenta detectar automáticamente si está separado por:
    - espacios
    - tabulaciones
    - comas
    - punto y coma
    """

    try:

        datos = pd.read_csv(
            archivo,
            sep=None,
            engine="python",
            comment="#"
        )

    except Exception:

        datos = pd.read_csv(
            archivo,
            sep=r"\s+",
            engine="python",
            comment="#",
            header=None
        )

    # --------------------------------------------------------------
    # Si tiene exactamente 6 columnas
    # --------------------------------------------------------------

    if len(datos.columns) == 6:

        # Si parecen datos sin encabezado
        try:

            pd.to_numeric(
                datos.iloc[:, 1]
            )

            datos.columns = COLUMNAS

        except Exception:

            datos.columns = COLUMNAS

    else:

        # Intentar nuevamente sin encabezado
        datos = pd.read_csv(
            archivo,
            sep=r"\s+|,|;",
            engine="python",
            comment="#",
            header=None
        )

        if len(datos.columns) != 6:

            raise ValueError(
                f"El archivo {archivo} no contiene "
                f"exactamente 6 columnas."
            )

        datos.columns = COLUMNAS

    # --------------------------------------------------------------
    # Convertir tiempo
    # --------------------------------------------------------------

    datos["tiempo"] = convertir_tiempo(
        datos["tiempo"]
    )

    # --------------------------------------------------------------
    # Convertir variables numéricas
    # --------------------------------------------------------------

    for columna in COLUMNAS[1:]:

        datos[columna] = pd.to_numeric(
            datos[columna],
            errors="coerce"
        )

    # --------------------------------------------------------------
    # Eliminar datos inválidos
    # --------------------------------------------------------------

    datos = datos.dropna(
        subset=[
            "tiempo",
            "irradiancia"
        ]
    ).copy()

    # --------------------------------------------------------------
    # Ordenar cronológicamente
    # --------------------------------------------------------------

    datos = datos.sort_values(
        "tiempo"
    ).reset_index(
        drop=True
    )

    return datos


# =====================================================================
# CLASIFICACIÓN SEGÚN EL CRITERIO DEL DIRECTOR
# =====================================================================

def clasificar_dia(irradiancia_promedio):

    """
    Clasifica el día utilizando los rangos indicados:

    0 - 200 W/m²       -> Nublado
    200 - 400 W/m²     -> Seminublado
    > 400 W/m²         -> Claro
    """

    if irradiancia_promedio < 200:

        return "Nublado"

    elif irradiancia_promedio <= 400:

        return "Seminublado"

    else:

        return "Claro"


# =====================================================================
# INTERVALO DE IRRADIANCIA SIGNIFICATIVA
# =====================================================================

def encontrar_periodo_significativo(datos):

    """
    Define irradiancia significativa como:

        G >= 0.10 * Gmax

    y busca el intervalo continuo que contiene
    el máximo de irradiancia.
    """

    G = datos["irradiancia"].values

    Gmax = np.max(G)

    umbral = UMBRAL_SIGNIFICATIVO * Gmax

    mascara = G >= umbral

    grupos = []

    inicio = None

    for i, valor in enumerate(mascara):

        if valor and inicio is None:

            inicio = i

        elif not valor and inicio is not None:

            fin = i - 1

            if (
                fin - inicio + 1
                >= MIN_PUNTOS_SIGNIFICATIVOS
            ):

                grupos.append(
                    (inicio, fin)
                )

            inicio = None

    # Si el último punto pertenece al intervalo
    if inicio is not None:

        fin = len(mascara) - 1

        if (
            fin - inicio + 1
            >= MIN_PUNTOS_SIGNIFICATIVOS
        ):

            grupos.append(
                (inicio, fin)
            )

    if not grupos:

        return None, umbral

    # --------------------------------------------------------------
    # Buscar el grupo que contiene el máximo
    # --------------------------------------------------------------

    indice_max = np.argmax(G)

    for grupo in grupos:

        inicio, fin = grupo

        if inicio <= indice_max <= fin:

            return grupo, umbral

    # Si no se encuentra, utilizar el grupo más largo

    grupo_mayor = max(
        grupos,
        key=lambda x: x[1] - x[0]
    )

    return grupo_mayor, umbral


# =====================================================================
# ANALIZAR UN ARCHIVO
# =====================================================================

def analizar_archivo(archivo):

    datos = leer_archivo(archivo)

    G = datos["irradiancia"]

    # ================================================================
    # ESTADÍSTICAS DE TODO EL REGISTRO
    # ================================================================

    G_promedio = G.mean()

    G_max = G.max()

    G_min = G.min()

    G_std = G.std()

    # --------------------------------------------------------------
    # Coeficiente de variación
    # --------------------------------------------------------------

    if G_promedio != 0:

        CV = G_std / G_promedio

    else:

        CV = np.nan

    # ================================================================
    # MOMENTO DEL MÁXIMO
    # ================================================================

    indice_max = G.idxmax()

    tiempo_max = datos.loc[
        indice_max,
        "tiempo"
    ]

    # ================================================================
    # PERIODO SIGNIFICATIVO
    # ================================================================

    intervalo, umbral = encontrar_periodo_significativo(
        datos
    )

    if intervalo is not None:

        inicio, fin = intervalo

        tiempo_inicio = datos.loc[
            inicio,
            "tiempo"
        ]

        tiempo_fin = datos.loc[
            fin,
            "tiempo"
        ]

        duracion_horas = (
            tiempo_fin - tiempo_inicio
        ).total_seconds() / 3600

        # ----------------------------------------------------------
        # Promedio únicamente durante el periodo significativo
        # ----------------------------------------------------------

        datos_significativos = datos.loc[
            inicio:fin
        ]

        G_promedio_significativo = (
            datos_significativos["irradiancia"]
            .mean()
        )

    else:

        tiempo_inicio = pd.NaT

        tiempo_fin = pd.NaT

        duracion_horas = np.nan

        G_promedio_significativo = np.nan

    # ================================================================
    # CLASIFICACIÓN
    # ================================================================

    clasificacion = clasificar_dia(
        G_promedio_significativo
    )

    # ================================================================
    # RESULTADOS
    # ================================================================

    resultado = {

        "archivo": os.path.basename(
            archivo
        ),

        "irradiancia_promedio_24h_W_m2":
            G_promedio,

        "irradiancia_promedio_significativa_W_m2":
            G_promedio_significativo,

        "irradiancia_maxima_W_m2":
            G_max,

        "irradiancia_minima_W_m2":
            G_min,

        "desviacion_estandar_W_m2":
            G_std,

        "coeficiente_variacion":
            CV,

        "hora_irradiancia_maxima":
            tiempo_max,

        "umbral_significativo_W_m2":
            umbral,

        "inicio_irradiancia_significativa":
            tiempo_inicio,

        "fin_irradiancia_significativa":
            tiempo_fin,

        "duracion_irradiancia_significativa_h":
            duracion_horas,

        "clasificacion":
            clasificacion
    }

    # ================================================================
    # GENERAR GRÁFICA
    # ================================================================

    nombre = os.path.splitext(
        os.path.basename(archivo)
    )[0]

    plt.figure(
        figsize=(12, 6)
    )

    plt.plot(
        datos["tiempo"],
        datos["irradiancia"],
        linewidth=1.2,
        label="Irradiancia global"
    )

    # --------------------------------------------------------------
    # Máximo
    # --------------------------------------------------------------

    plt.scatter(
        tiempo_max,
        G_max,
        s=60,
        zorder=5,
        label=f"Máximo = {G_max:.1f} W/m²"
    )

    plt.annotate(
        f"{G_max:.1f} W/m²",
        xy=(
            tiempo_max,
            G_max
        ),
        xytext=(
            10,
            10
        ),
        textcoords="offset points"
    )

    # --------------------------------------------------------------
    # Umbral
    # --------------------------------------------------------------

    plt.axhline(
        umbral,
        linestyle="--",
        linewidth=1,
        label=(
            f"Umbral significativo = "
            f"{umbral:.1f} W/m²"
        )
    )

    # --------------------------------------------------------------
    # Periodo significativo
    # --------------------------------------------------------------

    if intervalo is not None:

        plt.axvspan(
            tiempo_inicio,
            tiempo_fin,
            alpha=0.15,
            label="Periodo de irradiancia significativa"
        )

    # --------------------------------------------------------------
    # Título
    # --------------------------------------------------------------

    plt.title(
        f"Irradiancia global - {nombre}\n"
        f"Clasificación: {clasificacion}"
    )

    plt.xlabel(
        "Tiempo"
    )

    plt.ylabel(
        "Irradiancia global (W/m²)"
    )

    plt.grid(
        True,
        alpha=0.3
    )

    plt.legend()

    plt.gcf().autofmt_xdate()

    plt.tight_layout()

    # --------------------------------------------------------------
    # Guardar PNG
    # --------------------------------------------------------------

    archivo_png = os.path.join(
        CARPETA_RESULTADOS,
        f"{nombre}_irradiancia.png"
    )

    plt.savefig(
        archivo_png,
        dpi=300,
        bbox_inches="tight"
    )

    # --------------------------------------------------------------
    # Guardar PDF
    # --------------------------------------------------------------

    archivo_pdf = os.path.join(
        CARPETA_RESULTADOS,
        f"{nombre}_irradiancia.pdf"
    )

    plt.savefig(
        archivo_pdf,
        bbox_inches="tight"
    )

    plt.close()

    # ================================================================
    # INFORME TXT
    # ================================================================

    archivo_txt = os.path.join(
        CARPETA_RESULTADOS,
        f"{nombre}_analisis.txt"
    )

    with open(
        archivo_txt,
        "w",
        encoding="utf-8"
    ) as f:

        f.write(
            "====================================================\n"
        )

        f.write(
            "ANÁLISIS DE IRRADIANCIA\n"
        )

        f.write(
            "====================================================\n\n"
        )

        f.write(
            f"Archivo: {nombre}\n\n"
        )

        f.write(
            "ESTADÍSTICAS DEL REGISTRO\n"
        )

        f.write(
            "----------------------------------------------------\n"
        )

        f.write(
            f"Irradiancia promedio (24 h): "
            f"{G_promedio:.3f} W/m²\n"
        )

        f.write(
            f"Irradiancia máxima: "
            f"{G_max:.3f} W/m²\n"
        )

        f.write(
            f"Irradiancia mínima: "
            f"{G_min:.3f} W/m²\n"
        )

        f.write(
            f"Desviación estándar: "
            f"{G_std:.3f} W/m²\n"
        )

        f.write(
            f"Coeficiente de variación: "
            f"{CV:.4f}\n\n"
        )

        f.write(
            "MÁXIMO DE IRRADIANCIA\n"
        )

        f.write(
            "----------------------------------------------------\n"
        )

        f.write(
            f"Hora del máximo: "
            f"{tiempo_max}\n"
        )

        f.write(
            f"Irradiancia máxima: "
            f"{G_max:.3f} W/m²\n\n"
        )

        f.write(
            "IRRADIANCIA SIGNIFICATIVA\n"
        )

        f.write(
            "----------------------------------------------------\n"
        )

        f.write(
            f"Umbral: "
            f"{umbral:.3f} W/m²\n"
        )

        f.write(
            f"Inicio: "
            f"{tiempo_inicio}\n"
        )

        f.write(
            f"Fin: "
            f"{tiempo_fin}\n"
        )

        f.write(
            f"Duración: "
            f"{duracion_horas:.3f} h\n"
        )

        f.write(
            f"Promedio durante periodo significativo: "
            f"{G_promedio_significativo:.3f} W/m²\n\n"
        )

        f.write(
            "CLASIFICACIÓN\n"
        )

        f.write(
            "----------------------------------------------------\n"
        )

        f.write(
            f"{clasificacion}\n"
        )

        f.write(
            "\nCriterio utilizado:\n"
        )

        f.write(
            "0 <= G_promedio < 200 W/m² : Nublado\n"
        )

        f.write(
            "200 <= G_promedio <= 400 W/m² : Seminublado\n"
        )

        f.write(
            "G_promedio > 400 W/m² : Claro\n"
        )

    return resultado


# =====================================================================
# PROCESAR TODOS LOS ARCHIVOS
# =====================================================================

def main():

    archivos = sorted(
        glob.glob(
            os.path.join(
                CARPETA_DATOS,
                "*.dat"
            )
        )
    )

    if not archivos:

        print(
            "No se encontraron archivos .dat."
        )

        return

    print(
        f"\nSe encontraron "
        f"{len(archivos)} archivos .dat.\n"
    )

    resultados = []

    for archivo in archivos:

        print(
            f"Analizando: "
            f"{os.path.basename(archivo)}"
        )

        try:

            resultado = analizar_archivo(
                archivo
            )

            resultados.append(
                resultado
            )

            print(
                f"  Promedio 24 h: "
                f"{resultado['irradiancia_promedio_24h_W_m2']:.2f} W/m²"
            )

            print(
                f"  Máximo: "
                f"{resultado['irradiancia_maxima_W_m2']:.2f} W/m²"
            )

            print(
                f"  Clasificación: "
                f"{resultado['clasificacion']}"
            )

            print()

        except Exception as e:

            print(
                f"  ERROR: {e}\n"
            )

    # ================================================================
    # RESUMEN GENERAL
    # ================================================================

    if resultados:

        resumen = pd.DataFrame(
            resultados
        )

        archivo_csv = os.path.join(
            CARPETA_RESULTADOS,
            "resumen_irradiancia.csv"
        )

        resumen.to_csv(
            archivo_csv,
            index=False,
            encoding="utf-8"
        )

        print(
            "===================================================="
        )

        print(
            "ANÁLISIS FINALIZADO"
        )

        print(
            "===================================================="
        )

        print(
            f"Resultados: {CARPETA_RESULTADOS}/"
        )

        print(
            f"Resumen: {archivo_csv}"
        )


# =====================================================================
# EJECUTAR
# =====================================================================

if __name__ == "__main__":

    main()


Se encontraron 21 archivos .dat.

Analizando: 2026-07-11.dat


/tmp/ipykernel_5252/210151904.py:98: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  tiempo = pd.to_datetime(


  Promedio 24 h: 143.16 W/m²
  Máximo: 1278.67 W/m²
  Clasificación: Claro

Analizando: 2026-07-15.dat


/tmp/ipykernel_5252/210151904.py:98: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  tiempo = pd.to_datetime(


  Promedio 24 h: 134.51 W/m²
  Máximo: 1163.38 W/m²
  Clasificación: Seminublado

Analizando: 2026-07-16.dat


...

  Promedio 24 h: 109.79 W/m²
  Máximo: 1292.11 W/m²
  Clasificación: Seminublado

Analizando: 2026-08-15.dat


/tmp/ipykernel_5252/210151904.py:98: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  tiempo = pd.to_datetime(


  Promedio 24 h: 165.93 W/m²
  Máximo: 1403.16 W/m²
  Clasificación: Claro

Analizando: 2026-08-16.dat


/tmp/ipykernel_5252/210151904.py:98: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  tiempo = pd.to_datetime(


  Promedio 24 h: 120.23 W/m²
  Máximo: 999.87 W/m²
  Clasificación: Seminublado

ANÁLISIS FINALIZADO
Resultados: analisis_irradiancia/
Resumen: analisis_irradiancia/resumen_irradiancia.csv


In [ ]:
import os
import glob
import pandas as pd
from datetime import datetime, timedelta

def analizar_archivo(ruta_archivo):
    try:
        # Leer el archivo con separador de espacios, sin encabezados
        # Col 0: Tiempo, Col 1: Irradiancia (Radiación Global), etc.
        df = pd.read_csv(ruta_archivo, sep=r'\s+', header=None, engine='python')

        if df.shape[1] < 2:
            return None

        # Extraer columnas clave
        col_tiempo = df.iloc[:, 0]
        col_irradiancia = pd.to_numeric(df.iloc[:, 1], errors='coerce')

        # Crear un DataFrame temporal de trabajo
        df_trabajo = pd.DataFrame({'tiempo_str': col_tiempo, 'irradiancia': col_irradiancia})
        df_trabajo = df_trabajo.dropna()

        if df_trabajo.empty:
            return None

        # Convertir la columna de tiempo a formato datetime (asumiendo formato HH:MM:SS)
        # Usamos una fecha base ficticia solo para operar horas limpiamente
        df_trabajo['dt'] = pd.to_datetime(df_trabajo['tiempo_str'], format='%H:%M:%S', errors='coerce')
        df_trabajo = df_trabajo.dropna(subset=['dt'])

        if df_trabajo.empty:
            return None

        # 1. Definir el periodo significativo de irradiancia (ej: irradiancia > 10 W/m²)
        umbral_significativo = 10.0
        df_significativo = df_trabajo[df_trabajo['irradiancia'] > umbral_significativo]

        if df_significativo.empty:
            # Si no supera el umbral, tomamos todo el set disponible
            df_significativo = df_trabajo

        hora_inicio = df_significativo['tiempo_str'].iloc[0]
        hora_fin = df_significativo['tiempo_str'].iloc[-1]

        # Calcular el total de horas del periodo significativo
        t_inicio_dt = df_significativo['dt'].iloc[0]
        t_fin_dt = df_significativo['dt'].iloc[-1]
        total_segundos = (t_fin_dt - t_inicio_dt).total_seconds()
        total_horas = total_segundos / 3600.0

        # 2. Máximos y mínimos dentro del periodo significativo
        val_max = df_significativo['irradiancia'].max()
        idx_max = df_significativo['irradiancia'].idxmax()
        hora_max = df_significativo.loc[idx_max, 'tiempo_str']

        val_min = df_significativo['irradiancia'].min()
        idx_min = df_significativo['irradiancia'].idxmin()
        hora_min = df_significativo.loc[idx_min, 'tiempo_str']

        # 3. Intervalo de mayor irradiancia (Ventana donde se concentra el 90% o el pico máximo,
        # por ejemplo, el intervalo de 1 hora alrededor del pico máximo o las horas pico)
        # Aquí definiremos el intervalo de la hora pico más alta registrada:
        pico_dt = df_significativo.loc[idx_max, 'dt']
        ventana_inicio = (pico_dt - timedelta(minutes=30)).strftime('%H:%M:%S')
        ventana_fin = (pico_dt + timedelta(minutes=30)).strftime('%H:%M:%S')

        nombre_archivo = os.path.basename(ruta_archivo)

        resultado = {
            "archivo": nombre_archivo,
            "periodo_inicio": hora_inicio,
            "periodo_fin": hora_fin,
            "total_horas_periodo": round(total_horas, 2),
            "irradiancia_max": round(val_max, 2),
            "hora_max": hora_max,
            "irradiancia_min": round(val_min, 2),
            "hora_min": hora_min,
            "intervalo_pico_inicio": ventana_inicio,
            "intervalo_pico_fin": ventana_fin
        }

        return resultado

    except Exception as e:
        print(f"[ERROR] No se pudo procesar {ruta_archivo}: {e}")
        return None

def main():
    # Buscar todos los archivos .dat (puedes cambiarlo a .txt si prefieres)
    archivos = glob.glob("*.dat")

    if not archivos:
        print("No se encontraron archivos .dat en el directorio actual.")
        return

    resumen_global = []

    print("Procesando archivos y calculando parámetros de irradiancia...\n")
    for archivo in sorted(archivos):
        res = analizar_archivo(archivo)
        if res:
            resumen_global.append(res)
            print(f"[OK] Analizado: {archivo}")

    if not resumen_global:
        print("No se obtuvieron datos válidos para generar el resumen.")
        return

    # Crear el archivo de salida general o resumen
    archivo_salida = "resumen_irradiancia_general.txt"

    with open(archivo_salida, 'w', encoding='utf-8') as f:
        f.write("=" * 95 + "\n")
        f.write(" RESUMEN GENERAL - ANALISIS DE IRRADIANCIA (TRABAJO DE GRADO)\n")
        f.write("=" * 95 + "\n\n")

        for r in resumen_global:
            f.write(f"Archivo de Registro: {r['archivo']}\n")
            f.write(f"  > Periodo Significativo   : Desde las {r['periodo_inicio']} hasta las {r['periodo_fin']} (Duración: {r['total_horas_periodo']} h)\n")
            f.write(f"  > Irradiancia Máxima      : {r['irradiancia_max']} W/m² (Registrada a las {r['hora_max']})\n")
            f.write(f"  > Irradiancia Mínima      : {r['irradiancia_min']} W/m² (Registrada a las {r['hora_min']})\n")
            f.write(f"  > Intervalo de Mayor Pico : {r['intervalo_pico_inicio']} a {r['intervalo_pico_fin']}\n")
            f.write("-" * 95 + "\n")

    print(f"\n[¡ÉXITO!] Se ha generado el archivo resumen global: '{archivo_salida}'")

if __name__ == "__main__":
    main()

Procesando archivos y calculando parámetros de irradiancia...

[OK] Analizado: 2026-07-11.dat
[OK] Analizado: 2026-07-15.dat
[OK] Analizado: 2026-07-16.dat
[OK] Analizado: 2026-07-17.dat
[OK] Analizado: 2026-07-18.dat
[OK] Analizado: 2026-07-19.dat
[OK] Analizado: 2026-07-22.dat
[OK] Analizado: 2026-07-23.dat
[OK] Analizado: 2026-07-24.dat
[OK] Analizado: 2026-07-25.dat
[OK] Analizado: 2026-07-26.dat
[OK] Analizado: 2026-07-27.dat
[OK] Analizado: 2026-07-28.dat
[OK] Analizado: 2026-08-08.dat
[OK] Analizado: 2026-08-09.dat
[OK] Analizado: 2026-08-10.dat
[OK] Analizado: 2026-08-12.dat
[OK] Analizado: 2026-08-13.dat
[OK] Analizado: 2026-08-14.dat
[OK] Analizado: 2026-08-15.dat
[OK] Analizado: 2026-08-16.dat

[¡ÉXITO!] Se ha generado el archivo resumen global: 'resumen_irradiancia_general.txt'


In [ ]:
import numpy as np
import pandas as pd

# Lista de tus archivos .dat correspondientes a las condiciones atmosféricas
# Asegúrate de ajustar los nombres según corresponda a cada día (ej. claro, seminublado, nublado)
archivos = {
    'Claro (Ej: 2026-08-10)': '2026-08-10.dat',
    'Seminublado (Ej: 2026-07-24)': '2026-07-24.dat',
    'Nublado (Ej: 2026-07-23)': '2026-07-23.dat',
}

# Columnas según tu descripción:
# 0: tiempo, 1: irradiancia (G), 2: temp_amb, 3: temp_sis, 4: voltaje_pv (VPV), 5: voltaje_bat (VB)
column_names = ['tiempo', 'G', 'temp_amb', 'temp_sis', 'VPV', 'VB']

for condicion, filepath in archivos.iterrows() if isinstance(
    archivos, pd.Series
) else archivos.items():
  print(f'=== ANÁLISIS PARA CONDICIÓN: {condicion} ===')
  try:
    # Cargar datos (asumiendo espacio o tabulador como delimitador)
    df = pd.read_csv(filepath, delim_whitespace=True, header=None, names=column_names)

    # Convertir tiempo a formato datetime o string legible si es necesario
    # Cálculo de métricas para VPV
    VPV_max = df['VPV'].max()
    VPV_min = df['VPV'].min()
    idx_max_g = df['G'].idxmax()
    hora_VPV_max = df.loc[df['VPV'].idxmax(), 'tiempo']
    hora_G_max = df.loc[idx_max_g, 'tiempo']
    VPV_en_G_max = df.loc[idx_max_g, 'VPV']
    diff_VPV = VPV_max - VPV_min

    # Cálculo de métricas para VB y rangos
    VB_max = df['VB'].max()
    VB_min = df['VB'].min()
    VB_promedio = df['VB'].mean()
    delta_VB = VB_max - VB_min
    delta_VPV = VPV_max - VPV_min

    # Coeficiente de correlación de Pearson entre Irradiancia (G) y Voltaje PV (VPV)
    r_g_vpv = df['G'].corr(df['VPV'], method='pearson')

    print(f'  - Irradiancia Máxima (G_max) en hora: {hora_G_max}')
    print(f'  - VPV,max: {VPV_max:.3f} V (registrado a las {hora_VPV_max})')
    print(f'  - VPV,min: {VPV_min:.3f} V')
    print(f'  - Diferencia (VPV,max - VPV,min): {diff_VPV:.3f} V')
    print(f'  - VPV en el momento de G_max: {VPV_en_G_max:.3f} V')
    print(f'  - Coeficiente de correlación (r G, VPV): {r_g_vpv:.3f}')
    print(f'  - VB promedio: {VB_promedio:.3f} V')
    print(f'  - Delta VB (VB,max - VB,min): {delta_VB:.3f} V')
    print(f'  - Delta VPV: {delta_VPV:.3f} V\n')

  except FileNotFoundError:
    print(
        f'  [Aviso] Archivo {filepath} no encontrado. Coloca la ruta correcta'
        ' para este día.\n'
    )

=== ANÁLISIS PARA CONDICIÓN: Claro (Ej: 2026-08-10) ===
  - Irradiancia Máxima (G_max) en hora: 11:39:14
  - VPV,max: 20.583 V (registrado a las 14:54:14)
  - VPV,min: 0.001 V
  - Diferencia (VPV,max - VPV,min): 20.582 V
  - VPV en el momento de G_max: 19.757 V
  - Coeficiente de correlación (r G, VPV): 0.807
  - VB promedio: 12.770 V
  - Delta VB (VB,max - VB,min): 1.159 V
  - Delta VPV: 20.582 V

=== ANÁLISIS PARA CONDICIÓN: Seminublado (Ej: 2026-07-24) ===
  - Irradiancia Máxima (G_max) en hora: 12:59:48
  - VPV,max: 20.748 V (registrado a las 10:03:48)
  - VPV,min: 0.001 V
  - Diferencia (VPV,max - VPV,min): 20.747 V
  - VPV en el momento de G_max: 19.925 V
  - Coeficiente de correlación (r G, VPV): 0.720
  - VB promedio: 12.793 V
  - Delta VB (VB,max - VB,min): 1.200 V
  - Delta VPV: 20.747 V

=== ANÁLISIS PARA CONDICIÓN: Nublado (Ej: 2026-07-23) ===
  - Irradiancia Máxima (G_max) en hora: 13:38:08
  - VPV,max: 21.005 V (registrado a las 13:14:08)
  - VPV,min: 0.000 V
  - Diferenc

/tmp/ipykernel_2167/4190086914.py:22: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(filepath, delim_whitespace=True, header=None, names=column_names)
/tmp/ipykernel_2167/4190086914.py:22: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(filepath, delim_whitespace=True, header=None, names=column_names)
/tmp/ipykernel_2167/4190086914.py:22: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(filepath, delim_whitespace=True, header=None, names=column_names)
